## **Canada - Brazil Trade Report**


# Data Loading

In [1]:
# ============================================================================
# 00_data_loading.py  (COLAB CELL)
# Final Data Loading Cell
# Canada-Brazil Trade Project
# ============================================================================

import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

# ============================================================================
# GOOGLE DRIVE
# ============================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ============================================================================
# PATHS
# ============================================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/canada-brazil-trade-report")
DATA_FOLDER = PROJECT_ROOT / "data"
RAW_FOLDER = DATA_FOLDER / "raw"
REPORTS_FOLDER = PROJECT_ROOT / "reports"

RAW_FOLDER.mkdir(parents=True, exist_ok=True)
REPORTS_FOLDER.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("00 DATA LOADING")
print("=" * 80)
print(f"Project root : {PROJECT_ROOT}")
print(f"Raw folder   : {RAW_FOLDER}")
print(f"Reports      : {REPORTS_FOLDER}")

# ============================================================================
# HELPERS
# ============================================================================

def find_file(folder, keyword):
    """
    Find the first file whose name contains the keyword.
    Looks for CSV first, then XLSX.
    """
    folder = Path(folder)

    csv_matches = sorted(folder.glob("*.csv"))
    xlsx_matches = sorted(folder.glob("*.xlsx"))

    all_files = csv_matches + xlsx_matches
    matches = [f for f in all_files if keyword.lower() in f.name.lower()]

    if not matches:
        raise FileNotFoundError(
            f"No file found for keyword '{keyword}' in {folder}\n"
            f"Available files: {[f.name for f in all_files]}"
        )

    return matches[0]


def load_trade_file(path, flow_label):
    """
    Load StatsCan-style trade file.
    Handles both CSV and Excel.
    """
    path = Path(path)

    print(f"\nLoading {flow_label}: {path.name}")

    if path.suffix.lower() == ".csv":
        df = pd.read_csv(
            path,
            skiprows=1,
            skipfooter=3,
            engine="python",
            encoding="utf-8-sig"
        )
    elif path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(
            path,
            skiprows=1,
            skipfooter=3,
            engine="openpyxl"
        )
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    df["flow"] = flow_label
    return df


def clean_footer_rows(df):
    """
    Keep only rows with valid Period format YYYY-MM-DD.
    """
    if "Period" not in df.columns:
        raise KeyError("Expected column 'Period' not found in dataset.")

    valid_mask = df["Period"].astype(str).str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)
    removed = (~valid_mask).sum()
    df = df[valid_mask].copy().reset_index(drop=True)

    return df, int(removed)


def add_basic_features(df):
    """
    Add standard date/value/quantity columns used later.
    """
    df = df.copy()

    df["Period"] = pd.to_datetime(df["Period"], errors="coerce")
    df["year"] = df["Period"].dt.year
    df["month"] = df["Period"].dt.month
    df["month_year"] = df["Period"].dt.to_period("M").astype(str)

    if "Value ($)" in df.columns:
        df["Value ($)"] = pd.to_numeric(df["Value ($)"], errors="coerce").fillna(0)

    if "Quantity" in df.columns:
        df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce").fillna(0)
    else:
        df["Quantity"] = 0

    if "Commodity" in df.columns:
        commodity_str = df["Commodity"].astype(str)

        df["commodity_code"] = commodity_str.str.extract(r"^(\d+)", expand=False)
        df["commodity_code"] = df["commodity_code"].fillna("").astype(str).str.zfill(10)

        df["Commodity"] = commodity_str.str.strip()

    if "Value ($)" in df.columns and "Quantity" in df.columns:
        df["unit_price"] = np.where(
            df["Quantity"] > 0,
            df["Value ($)"] / df["Quantity"],
            np.nan
        )
    else:
        df["unit_price"] = np.nan

    return df


def print_overview(df, name):
    print("\n" + "-" * 80)
    print(name)
    print("-" * 80)
    print(f"Shape: {df.shape}")
    print("Columns:")
    print(df.columns.tolist())
    print("\nHead:")
    print(df.head())


# ============================================================================
# LOAD FILES
# ============================================================================

IMPORT_FILE = find_file(RAW_FOLDER, "import")
EXPORT_FILE = find_file(RAW_FOLDER, "export")

df_import = load_trade_file(IMPORT_FILE, "Import")
df_export = load_trade_file(EXPORT_FILE, "Export")

df_import, removed_import = clean_footer_rows(df_import)
df_export, removed_export = clean_footer_rows(df_export)

df_import = add_basic_features(df_import)
df_export = add_basic_features(df_export)

print(f"\nFooter rows removed — Import: {removed_import}")
print(f"Footer rows removed — Export: {removed_export}")

print_overview(df_import, "IMPORT DATA")
print_overview(df_export, "EXPORT DATA")

print("\n" + "=" * 80)
print("✅ DATA LOADING COMPLETE")
print("=" * 80)
print(f"Import rows: {len(df_import):,}")
print(f"Export rows: {len(df_export):,}")

ModuleNotFoundError: No module named 'google.colab'

# Data Understanding

In [ ]:
# ============================================================================
# 01_data_understanding.py  (COLAB CELL)
# Final Data Understanding & PDF Profiling Report
# Canada-Brazil Trade Project
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import textwrap
from datetime import datetime
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages

warnings.filterwarnings("ignore")
plt.rcParams["figure.max_open_warning"] = 100

# ============================================================================
# PATHS
# ============================================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/canada-brazil-trade-report")
REPORTS_FOLDER = PROJECT_ROOT / "reports"
REPORTS_FOLDER.mkdir(parents=True, exist_ok=True)

PDF_FILE = REPORTS_FOLDER / "01_data_understanding_report.pdf"
IMPORT_SUMMARY_FILE = REPORTS_FOLDER / "summary_import_data.csv"
EXPORT_SUMMARY_FILE = REPORTS_FOLDER / "summary_export_data.csv"

print("=" * 80)
print("01 DATA UNDERSTANDING")
print("=" * 80)
print(f"Reports folder : {REPORTS_FOLDER}")
print(f"PDF output     : {PDF_FILE}")

# ============================================================================
# CHECK INPUT DATA
# ============================================================================

if "df_import" not in globals() or "df_export" not in globals():
    raise NameError(
        "df_import and df_export were not found.\n"
        "Run the DATA LOADING cell first."
    )

# ============================================================================
# BASIC ANALYSIS FUNCTIONS
# ============================================================================

def basic_info(df, name):
    print("\n" + "=" * 80)
    print(f"BASIC INFO — {name}")
    print("=" * 80)
    print("\nShape:")
    print(df.shape)
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nData Types:")
    print(df.dtypes)
    print("\nSample:")
    print(df.head())


def missing_values(df, name):
    print("\n" + "=" * 80)
    print(f"MISSING VALUES — {name}")
    print("=" * 80)

    missing = df.isna().sum()
    missing_pct = (missing / len(df)) * 100 if len(df) > 0 else 0

    missing_df = pd.DataFrame({
        "column": missing.index,
        "missing_count": missing.values,
        "missing_pct": missing_pct.values if hasattr(missing_pct, "values") else missing_pct
    }).sort_values(by="missing_pct", ascending=False)

    missing_df = missing_df[missing_df["missing_count"] > 0]

    if missing_df.empty:
        print("No missing values found.")
    else:
        print(missing_df.to_string(index=False))

    return missing_df


def numerical_summary(df, name):
    print("\n" + "=" * 80)
    print(f"NUMERICAL SUMMARY — {name}")
    print("=" * 80)

    num_cols = df.select_dtypes(include=np.number).columns

    if len(num_cols) == 0:
        print("No numerical columns found.")
        return pd.DataFrame()

    summary = df[num_cols].describe().T
    print(summary)
    return summary


def categorical_summary(df, name):
    print("\n" + "=" * 80)
    print(f"CATEGORICAL SUMMARY — {name}")
    print("=" * 80)

    cat_cols = df.select_dtypes(include="object").columns
    records = []

    if len(cat_cols) == 0:
        print("No categorical columns found.")
        return pd.DataFrame()

    for col in cat_cols:
        vc = df[col].astype(str).value_counts(dropna=False)
        top_value = vc.index[0] if len(vc) > 0 else "N/A"
        top_freq = vc.iloc[0] if len(vc) > 0 else 0

        print(f"\n🔹 {col}")
        print(vc.head(10))

        records.append({
            "column": col,
            "unique_values": df[col].nunique(dropna=True),
            "top_value": top_value,
            "top_frequency": int(top_freq)
        })

    return pd.DataFrame(records)


def time_analysis(df, name):
    print("\n" + "=" * 80)
    print(f"TIME ANALYSIS — {name}")
    print("=" * 80)

    result = {}

    if "Period" in df.columns and df["Period"].notna().any():
        result["min_date"] = df["Period"].min()
        result["max_date"] = df["Period"].max()

        print("Date Range:")
        print(result["min_date"], "→", result["max_date"])

        if "year" in df.columns:
            yearly = df["year"].value_counts().sort_index()
            print("\nRecords per Year:")
            print(yearly)
            result["records_per_year"] = yearly

        if "month_year" in df.columns:
            monthly = df["month_year"].value_counts().sort_index()
            print("\nRecords per Month:")
            print(monthly.head(12))
            result["records_per_month"] = monthly
    else:
        print("No valid 'Period' column found.")

    return result


def business_metrics(df, name):
    print("\n" + "=" * 80)
    print(f"BUSINESS METRICS — {name}")
    print("=" * 80)

    metrics = {}

    if "Value ($)" in df.columns:
        total_value = df["Value ($)"].sum()
        print(f"Total Trade Value: ${total_value:,.2f}")
        metrics["total_trade_value"] = total_value

    if "Quantity" in df.columns:
        total_qty = df["Quantity"].sum()
        zero_qty = ((df["Quantity"] == 0) | (df["Quantity"].isna())).sum()
        zero_pct = (((df["Quantity"] == 0) | (df["Quantity"].isna())).mean() * 100) if len(df) > 0 else 0

        print(f"Total Quantity: {total_qty:,.0f}")
        print(f"Zero / Missing Quantity Rows: {zero_qty:,} ({zero_pct:.2f}%)")

        metrics["total_quantity"] = total_qty
        metrics["zero_quantity_rows"] = int(zero_qty)
        metrics["zero_quantity_pct"] = round(zero_pct, 2)

    if "unit_price" in df.columns:
        unit_price_clean = df["unit_price"].replace([np.inf, -np.inf], np.nan).dropna()
        unit_price_clean = unit_price_clean[unit_price_clean > 0]

        if len(unit_price_clean) > 0:
            print("\nUnit Price Stats:")
            print(unit_price_clean.describe())

            metrics["median_unit_price"] = unit_price_clean.median()
            metrics["mean_unit_price"] = unit_price_clean.mean()
            metrics["max_unit_price"] = unit_price_clean.max()

    return metrics


def top_products(df, name):
    print("\n" + "=" * 80)
    print(f"TOP PRODUCTS — {name}")
    print("=" * 80)

    if "Commodity" in df.columns and "Value ($)" in df.columns:
        top = (
            df.groupby("Commodity")["Value ($)"]
            .sum()
            .sort_values(ascending=False)
            .head(10)
        )
        print(top)
        return top

    print("No 'Commodity' or 'Value ($)' column found.")
    return pd.Series(dtype=float)


def duplicate_analysis(df, name):
    print("\n" + "=" * 80)
    print(f"DUPLICATE ANALYSIS — {name}")
    print("=" * 80)

    dup_count = int(df.duplicated().sum())
    dup_pct = round((df.duplicated().mean() * 100), 2) if len(df) > 0 else 0.0

    print(f"Duplicate rows: {dup_count:,} ({dup_pct}%)")

    return {
        "duplicate_rows": dup_count,
        "duplicate_pct": dup_pct
    }


def outlier_analysis(df, name):
    print("\n" + "=" * 80)
    print(f"OUTLIER ANALYSIS — {name}")
    print("=" * 80)

    if "unit_price" not in df.columns:
        print("No unit_price column found.")
        return {"outlier_count": 0, "outlier_pct": 0}

    s = df["unit_price"].replace([np.inf, -np.inf], np.nan).dropna()
    s = s[s > 0]

    if len(s) < 4:
        print("Not enough valid unit_price data for outlier analysis.")
        return {"outlier_count": 0, "outlier_pct": 0}

    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = max(q1 - 1.5 * iqr, 0)
    upper = q3 + 1.5 * iqr

    outliers = ((s < lower) | (s > upper)).sum()
    outlier_pct = round((outliers / len(s)) * 100, 2)

    print(f"Outliers in unit_price: {outliers:,} ({outlier_pct}%)")
    print(f"Bounds: {lower:,.4f} to {upper:,.4f}")

    return {
        "outlier_count": int(outliers),
        "outlier_pct": outlier_pct,
        "lower_bound": lower,
        "upper_bound": upper
    }


def build_profile(df, name):
    basic_info(df, name)
    profile = {
        "name": name,
        "shape": df.shape,
        "missing": missing_values(df, name),
        "numerical": numerical_summary(df, name),
        "categorical": categorical_summary(df, name),
        "time_info": time_analysis(df, name),
        "metrics": business_metrics(df, name),
        "top_products": top_products(df, name),
        "duplicates": duplicate_analysis(df, name),
        "outliers": outlier_analysis(df, name),
    }
    return profile


def save_summary(df, output_file):
    summary = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "non_null_count": [df[c].notna().sum() for c in df.columns],
        "null_count": [df[c].isna().sum() for c in df.columns],
        "null_pct": [round(df[c].isna().mean() * 100, 2) for c in df.columns],
        "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
    })
    summary.to_csv(output_file, index=False)
    print(f"✅ Saved summary: {output_file}")


# ============================================================================
# CHARTS
# ============================================================================

def chart_missing_values(profile):
    fig, ax = plt.subplots(figsize=(10, 6))
    missing_df = profile["missing"]

    if missing_df.empty:
        ax.text(0.5, 0.5, "No missing values", ha="center", va="center", fontsize=14)
        ax.axis("off")
        return fig

    chart_df = missing_df.sort_values("missing_pct", ascending=True)
    ax.barh(chart_df["column"], chart_df["missing_pct"], edgecolor="black")
    ax.set_title(f"Missing Values (%) — {profile['name']}", fontsize=14, fontweight="bold")
    ax.set_xlabel("Missing %")
    ax.grid(axis="x", linestyle="--", alpha=0.4)
    plt.tight_layout()
    return fig


def chart_top_products(profile):
    fig, ax = plt.subplots(figsize=(10, 6))
    top = profile["top_products"]

    if len(top) == 0:
        ax.text(0.5, 0.5, "No product/value data available", ha="center", va="center", fontsize=14)
        ax.axis("off")
        return fig

    top = top.sort_values(ascending=True)
    ax.barh(top.index.astype(str), top.values, edgecolor="black")
    ax.set_title(f"Top Products by Trade Value — {profile['name']}", fontsize=14, fontweight="bold")
    ax.set_xlabel("Trade Value ($)")
    ax.grid(axis="x", linestyle="--", alpha=0.4)
    plt.tight_layout()
    return fig


def chart_yearly_records(profile):
    fig, ax = plt.subplots(figsize=(10, 5))
    yearly = profile["time_info"].get("records_per_year", pd.Series(dtype=int))

    if len(yearly) == 0:
        ax.text(0.5, 0.5, "No yearly data available", ha="center", va="center", fontsize=14)
        ax.axis("off")
        return fig

    ax.bar(yearly.index.astype(str), yearly.values, edgecolor="black")
    ax.set_title(f"Records per Year — {profile['name']}", fontsize=14, fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel("Records")
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    return fig


def chart_monthly_records(profile):
    fig, ax = plt.subplots(figsize=(11, 5))
    monthly = profile["time_info"].get("records_per_month", pd.Series(dtype=int))

    if len(monthly) == 0:
        ax.text(0.5, 0.5, "No monthly data available", ha="center", va="center", fontsize=14)
        ax.axis("off")
        return fig

    monthly = monthly.sort_index()
    x = np.arange(len(monthly))

    ax.plot(x, monthly.values, marker="o", linewidth=2)
    step = max(1, len(monthly) // 8)
    ax.set_xticks(x[::step])
    ax.set_xticklabels(monthly.index[::step], rotation=45, ha="right")
    ax.set_title(f"Records per Month — {profile['name']}", fontsize=14, fontweight="bold")
    ax.set_ylabel("Records")
    ax.grid(alpha=0.4)
    plt.tight_layout()
    return fig


def chart_unit_price(profile, df):
    fig, ax = plt.subplots(figsize=(10, 6))

    if "unit_price" not in df.columns:
        ax.text(0.5, 0.5, "No unit_price column", ha="center", va="center", fontsize=14)
        ax.axis("off")
        return fig

    s = df["unit_price"].replace([np.inf, -np.inf], np.nan).dropna()
    s = s[s > 0]

    if len(s) == 0:
        ax.text(0.5, 0.5, "No valid unit_price data", ha="center", va="center", fontsize=14)
        ax.axis("off")
        return fig

    ax.hist(s, bins=50, edgecolor="black")
    ax.set_xscale("log")
    ax.set_title(f"Unit Price Distribution (log scale) — {profile['name']}", fontsize=14, fontweight="bold")
    ax.set_xlabel("Unit Price")
    ax.set_ylabel("Frequency")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    return fig


# ============================================================================
# PDF HELPERS
# ============================================================================

def add_wrapped_text(fig, x, y, text, width=95, fontsize=10.5, weight=None, color="black"):
    wrapped = textwrap.fill(str(text), width=width)
    fig.text(x, y, wrapped, fontsize=fontsize, fontweight=weight, color=color, va="top")
    lines = wrapped.count("\n") + 1
    return y - lines * 0.038


def add_chart_page(pdf, title, chart_fig, insights=None):
    page = plt.figure(figsize=(11.7, 8.3))
    page.text(0.5, 0.95, title, ha="center", fontsize=18, fontweight="bold")

    temp_path = "/tmp/temp_chart.png"
    chart_fig.savefig(temp_path, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(chart_fig)

    img = plt.imread(temp_path)
    ax = page.add_axes([0.07, 0.30, 0.86, 0.55])
    ax.imshow(img)
    ax.axis("off")

    y = 0.22
    if insights:
        page.text(0.07, y, "Key Insights", fontsize=12, fontweight="bold")
        y -= 0.04
        for insight in insights:
            y = add_wrapped_text(page, 0.09, y, f"• {insight}", fontsize=10)

    pdf.savefig(page, bbox_inches="tight", pad_inches=0.3)
    plt.close(page)


def create_pdf_report(df_import, df_export, import_profile, export_profile):
    print("\n" + "=" * 80)
    print("CREATING PDF REPORT")
    print("=" * 80)

    with PdfPages(PDF_FILE) as pdf:
        fig = plt.figure(figsize=(11.7, 8.3))
        fig.text(0.5, 0.68, "01 Data Understanding Report", ha="center", fontsize=26, fontweight="bold")
        fig.text(0.5, 0.58, "Canada ↔ Brazil Trade Project", ha="center", fontsize=18)
        fig.text(0.5, 0.48, "Imports and Exports Profiling", ha="center", fontsize=14, color="#444444")
        fig.text(0.5, 0.35, f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}", ha="center", fontsize=11, color="gray")
        plt.axis("off")
        pdf.savefig(fig, bbox_inches="tight", pad_inches=0.4)
        plt.close(fig)

        fig = plt.figure(figsize=(11.7, 8.3))
        fig.text(0.5, 0.93, "Executive Summary", ha="center", fontsize=20, fontweight="bold")

        y = 0.84
        lines = [
            f"Import dataset: {import_profile['shape'][0]:,} rows and {import_profile['shape'][1]} columns",
            f"Export dataset: {export_profile['shape'][0]:,} rows and {export_profile['shape'][1]} columns",
            f"Import zero-quantity rows: {import_profile['metrics'].get('zero_quantity_rows', 0):,} ({import_profile['metrics'].get('zero_quantity_pct', 0)}%)",
            f"Export zero-quantity rows: {export_profile['metrics'].get('zero_quantity_rows', 0):,} ({export_profile['metrics'].get('zero_quantity_pct', 0)}%)",
            f"Import duplicates: {import_profile['duplicates']['duplicate_rows']:,} ({import_profile['duplicates']['duplicate_pct']}%)",
            f"Export duplicates: {export_profile['duplicates']['duplicate_rows']:,} ({export_profile['duplicates']['duplicate_pct']}%)",
            f"Import unit_price outliers: {import_profile['outliers']['outlier_count']:,} ({import_profile['outliers']['outlier_pct']}%)",
            f"Export unit_price outliers: {export_profile['outliers']['outlier_count']:,} ({export_profile['outliers']['outlier_pct']}%)",
        ]

        for line in lines:
            y = add_wrapped_text(fig, 0.08, y, f"• {line}", fontsize=11)

        plt.axis("off")
        pdf.savefig(fig, bbox_inches="tight", pad_inches=0.4)
        plt.close(fig)

        for label, df, profile in [
            ("Import", df_import, import_profile),
            ("Export", df_export, export_profile),
        ]:
            add_chart_page(
                pdf,
                f"{label} — Missing Values",
                chart_missing_values(profile),
                insights=[
                    "Shows which columns need the most attention before analysis.",
                    "High missingness in key fields may affect business conclusions."
                ]
            )

            add_chart_page(
                pdf,
                f"{label} — Top Products by Trade Value",
                chart_top_products(profile),
                insights=[
                    "Highlights the most valuable products in the dataset.",
                    "These products should be prioritized during quality checks."
                ]
            )

            add_chart_page(
                pdf,
                f"{label} — Records per Year",
                chart_yearly_records(profile),
                insights=["Confirms temporal coverage at the yearly level."]
            )

            add_chart_page(
                pdf,
                f"{label} — Records per Month",
                chart_monthly_records(profile),
                insights=["Helps detect gaps, drops, or irregular monthly patterns."]
            )

            add_chart_page(
                pdf,
                f"{label} — Unit Price Distribution",
                chart_unit_price(profile, df),
                insights=[
                    "Log scale is used because unit prices usually have a long tail.",
                    "Very high or very low values should be checked before using averages."
                ]
            )

    print(f"✅ PDF saved: {PDF_FILE}")


# ============================================================================
# MAIN EXECUTION
# ============================================================================

print("\n" + "=" * 80)
print("RUNNING DATA UNDERSTANDING")
print("=" * 80)

print("\n🔍 DATA UNDERSTANDING — IMPORTS")
import_profile = build_profile(df_import, "IMPORT DATA")

print("\n🔍 DATA UNDERSTANDING — EXPORTS")
export_profile = build_profile(df_export, "EXPORT DATA")

print("\n📁 SAVING SUMMARY FILES")
save_summary(df_import, IMPORT_SUMMARY_FILE)
save_summary(df_export, EXPORT_SUMMARY_FILE)

print("\n🧾 CREATING PDF REPORT")
create_pdf_report(df_import, df_export, import_profile, export_profile)

print("\n" + "=" * 80)
print("✅ DATA UNDERSTANDING COMPLETE")
print("=" * 80)
print(f"PDF report         : {PDF_FILE}")
print(f"Import summary CSV : {IMPORT_SUMMARY_FILE}")
print(f"Export summary CSV : {EXPORT_SUMMARY_FILE}")

01 DATA UNDERSTANDING
Reports folder : /content/drive/MyDrive/canada-brazil-trade-report/reports
PDF output     : /content/drive/MyDrive/canada-brazil-trade-report/reports/01_data_understanding_report.pdf

RUNNING DATA UNDERSTANDING

🔍 DATA UNDERSTANDING — IMPORTS

BASIC INFO — IMPORT DATA

Shape:
(72320, 14)

Columns:
['Period', 'Commodity', 'Province', 'Country', 'State', 'Value ($)', 'Quantity', 'Unit of measure', 'flow', 'year', 'month', 'month_year', 'commodity_code', 'unit_price']

Data Types:
Period             datetime64[ns]
Commodity                  object
Province                   object
Country                    object
State                     float64
Value ($)                   int64
Quantity                    int64
Unit of measure            object
flow                       object
year                        int32
month                       int32
month_year                 object
commodity_code             object
unit_price                float64
dtype: object

Samp

# Data Cleaning

In [ ]:
# ============================================================================
# 02_data_cleaning.py  (COLAB CELL)
# Final Data Cleaning + PDF Audit Report
# Canada-Brazil Trade Project
# ============================================================================

import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "reportlab", "openpyxl", "Pillow"], check=False)

import re
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

# ============================================================================
# PATHS
# ============================================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/canada-brazil-trade-report")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
CLEAN_DIR = DATA_DIR / "clean"
REPORT_DIR = PROJECT_ROOT / "reports"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

MAPPING_PATH = DATA_DIR / "hs_mapping.xlsx"
UOM_REF_PATH = DATA_DIR / "t2026_uom.xlsx"

CLEAN_CSV = CLEAN_DIR / "dataset_clean.csv"
AUDIT_PDF = REPORT_DIR / "02_data_cleaning_report.pdf"

print("=" * 80)
print("02 DATA CLEANING")
print("=" * 80)
print(f"Clean CSV : {CLEAN_CSV}")
print(f"Audit PDF : {AUDIT_PDF}")

# ============================================================================
# CHECK INPUTS
# ============================================================================

if "df_import" not in globals() or "df_export" not in globals():
    raise NameError(
        "df_import and df_export were not found.\n"
        "Run the DATA LOADING cell first."
    )

# ============================================================================
# CONSTANTS
# ============================================================================

UOM_TEXT_TO_CODE = {
    "Weight in kilograms": "KGM",
    "Number": "NMB",
    "Weight in metric tonne": "TNE",
    "Volume in litres": "LTR",
    "Area in square metres": "MTK",
    "Length in metres": "MTR",
    "Metric tonne air dry": "TSD",
    "Volume in cubic metres": "MTQ",
    "Number in thousands": "MIL",
    "Volume in litres of pure alcohol": "LPA",
    "Number of dozens": "DZN",
    "Number of pairs": "PAR",
    "Weight in grams": "GRM",
    "Number of packages": "NMB",
    "Number of gross": "GRO",
    "Weight in kilograms of named substance": "KSD",
    "Weight in carats": "CTM",
    "Radioactivity in gigabecquerels": "GBQ",
    "Radioactivity in megabecquerels": "MBQ",
    "Blank": "BLANK",
}

IQR_MULTIPLIER = 1.5

FINAL_COLS = [
    "period_m", "year", "month", "quarter", "flow",
    "province", "country", "description",
    "hs2", "hs4", "hs6", "hs8", "hs10",
    "section", "section_name", "chapter_name",
    "value", "quantity",
    "uom_text", "uom_code", "uom_status",
    "has_quantity", "is_ch98", "is_suppressed", "is_outlier",
    "median_value_hs8", "lower_bound_hs8", "upper_bound_hs8"
]

# ============================================================================
# HELPERS
# ============================================================================

def clean_text(x):
    if pd.isna(x):
        return np.nan
    x = str(x)
    x = unicodedata.normalize("NFKD", x)
    x = x.encode("ascii", "ignore").decode("utf-8", errors="ignore")
    x = re.sub(r"\s+", " ", x).strip()
    return x


def extract_hs_parts(commodity_value):
    s = str(commodity_value) if pd.notna(commodity_value) else ""
    code = re.match(r"^(\d+)", s)
    code = code.group(1) if code else ""
    code = code.zfill(10) if code else ""

    desc = s
    if " - " in s:
        parts = s.split(" - ", 1)
        if len(parts) > 1:
            desc = parts[1].strip()

    hs2 = code[:2] if len(code) >= 2 else np.nan
    hs4 = code[:4] if len(code) >= 4 else np.nan
    hs6 = code[:6] if len(code) >= 6 else np.nan
    hs8 = code[:8] if len(code) >= 8 else np.nan
    hs10 = code[:10] if len(code) >= 10 else np.nan

    return hs2, hs4, hs6, hs8, hs10, clean_text(desc)


def load_mapping_file(path):
    if not Path(path).exists():
        print(f"⚠️ Mapping file not found: {path}")
        return pd.DataFrame(columns=["hs2", "section", "section_name", "chapter_name"])

    df_map = pd.read_excel(path)
    df_map.columns = [str(c).strip() for c in df_map.columns]

    rename_candidates = {
        "HS2": "hs2",
        "hs2": "hs2",
        "Section": "section",
        "section": "section",
        "Section Name": "section_name",
        "section_name": "section_name",
        "Chapter Name": "chapter_name",
        "chapter_name": "chapter_name",
    }
    df_map = df_map.rename(columns=rename_candidates)

    for col in ["hs2", "section", "section_name", "chapter_name"]:
        if col not in df_map.columns:
            df_map[col] = np.nan

    df_map["hs2"] = df_map["hs2"].astype(str).str.extract(r"(\d+)", expand=False).str.zfill(2)

    return df_map[["hs2", "section", "section_name", "chapter_name"]].drop_duplicates()


def build_uom_reference(path):
    if not Path(path).exists():
        print(f"⚠️ UOM reference file not found: {path}")
        return pd.DataFrame(columns=["hs6", "expected_uom_code"])

    try:
        uom_ref = pd.read_excel(path)
        uom_ref.columns = [str(c).strip() for c in uom_ref.columns]

        possible_hs = [c for c in uom_ref.columns if "hs" in c.lower()]
        possible_uom = [c for c in uom_ref.columns if "uom" in c.lower() or "unit" in c.lower()]

        if not possible_hs or not possible_uom:
            return pd.DataFrame(columns=["hs6", "expected_uom_code"])

        hs_col = possible_hs[0]
        uom_col = possible_uom[0]

        out = uom_ref[[hs_col, uom_col]].copy()
        out.columns = ["hs6", "expected_uom_code"]
        out["hs6"] = out["hs6"].astype(str).str.extract(r"(\d+)", expand=False).str.zfill(6)
        out["expected_uom_code"] = out["expected_uom_code"].astype(str).str.strip().str.upper()

        return out.drop_duplicates()

    except Exception as e:
        print(f"⚠️ Could not read UOM reference: {e}")
        return pd.DataFrame(columns=["hs6", "expected_uom_code"])


def compute_outlier_bounds(df):
    base = df.loc[df["value"] > 0, ["hs8", "value"]].dropna().copy()

    grouped = base.groupby("hs8")["value"]
    stats = grouped.agg(
        n="count",
        q1=lambda s: s.quantile(0.25),
        q3=lambda s: s.quantile(0.75),
        median_value_hs8="median"
    ).reset_index()

    stats["iqr"] = stats["q3"] - stats["q1"]
    stats["lower_bound_hs8"] = (stats["q1"] - IQR_MULTIPLIER * stats["iqr"]).clip(lower=0)
    stats["upper_bound_hs8"] = stats["q3"] + IQR_MULTIPLIER * stats["iqr"]

    stats.loc[stats["n"] < 4, ["median_value_hs8", "lower_bound_hs8", "upper_bound_hs8"]] = np.nan

    return stats[["hs8", "median_value_hs8", "lower_bound_hs8", "upper_bound_hs8"]]


def create_pdf_report(df, output_pdf):
    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(
        name="TitleCustom",
        fontSize=18,
        leading=22,
        alignment=TA_CENTER,
        textColor=colors.HexColor("#0D3B66"),
        spaceAfter=12
    ))

    doc = SimpleDocTemplate(str(output_pdf), pagesize=A4)
    story = []

    story.append(Paragraph("02 Data Cleaning Report", styles["TitleCustom"]))
    story.append(Paragraph("Canada ↔ Brazil Trade Project", styles["Heading2"]))
    story.append(Spacer(1, 0.4 * cm))

    summary_rows = [
        ["Metric", "Value"],
        ["Final rows", f"{len(df):,}"],
        ["Final columns", f"{len(df.columns)}"],
        ["Suppressed rows (value = 0)", f"{df['is_suppressed'].sum():,}"],
        ["Chapter 98 rows", f"{df['is_ch98'].sum():,}"],
        ["Rows with zero quantity", f"{(df['quantity'] == 0).sum():,}"],
        ["UOM errors", f"{(df['uom_status'] == 'Error').sum():,}"],
        ["UOM mismatches", f"{(df['uom_status'] == 'Mismatch').sum():,}"],
        ["Value outliers flagged", f"{df['is_outlier'].sum():,}"],
    ]

    tbl = Table(summary_rows, colWidths=[8 * cm, 7 * cm])
    tbl.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#0D3B66")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.whitesmoke, colors.lightgrey]),
        ("FONTSIZE", (0, 0), (-1, -1), 9),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
    ]))
    story.append(tbl)
    story.append(Spacer(1, 0.5 * cm))

    story.append(Paragraph("Final Dataset Columns", styles["Heading2"]))
    cols_text = "<br/>".join(df.columns.tolist())
    story.append(Paragraph(cols_text, styles["BodyText"]))
    story.append(PageBreak())

    story.append(Paragraph("Sample of Cleaned Dataset", styles["Heading2"]))
    sample = df.head(15).copy()

    sample_display = [sample.columns.tolist()] + sample.astype(str).values.tolist()
    sample_tbl = Table(sample_display, repeatRows=1)
    sample_tbl.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#0D3B66")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("GRID", (0, 0), (-1, -1), 0.25, colors.grey),
        ("FONTSIZE", (0, 0), (-1, -1), 6),
    ]))
    story.append(sample_tbl)

    doc.build(story)
    print(f"✅ PDF saved: {output_pdf}")


# ============================================================================
# MAIN CLEANING PIPELINE
# ============================================================================

print("► Combining import and export datasets...")
df = pd.concat([df_import.copy(), df_export.copy()], ignore_index=True)

print("► Standardizing column names and core fields...")
df["flow"] = df["flow"].astype(str)

for col in ["Province", "Country", "Unit of measure", "Commodity"]:
    if col not in df.columns:
        df[col] = np.nan

df["province"] = df["Province"].apply(clean_text)
df["country"] = df["Country"].apply(clean_text)
df["uom_text"] = df["Unit of measure"].apply(clean_text)

hs_parts = df["Commodity"].apply(extract_hs_parts)
df[["hs2", "hs4", "hs6", "hs8", "hs10", "description"]] = pd.DataFrame(hs_parts.tolist(), index=df.index)

print("► Creating date features...")
df["Period"] = pd.to_datetime(df["Period"], errors="coerce")
df["period_m"] = df["Period"].dt.to_period("M").astype(str)
df["year"] = df["Period"].dt.year
df["month"] = df["Period"].dt.month
df["quarter"] = df["Period"].dt.quarter

print("► Standardizing numeric columns...")
df["value"] = pd.to_numeric(df.get("Value ($)", 0), errors="coerce").fillna(0)
df["quantity"] = pd.to_numeric(df.get("Quantity", 0), errors="coerce").fillna(0)

print("► Adding core flags...")
df["uom_code"] = df["uom_text"].map(UOM_TEXT_TO_CODE).fillna("UNKNOWN")
df["has_quantity"] = df["uom_text"].fillna("").ne("Blank")
df["is_ch98"] = df["hs2"].astype(str) == "98"
df["is_suppressed"] = df["value"] == 0

print("► Joining HS mapping...")
mapping_df = load_mapping_file(MAPPING_PATH)
df = df.merge(mapping_df, on="hs2", how="left")

print("► Auditing UOM...")
uom_ref_df = build_uom_reference(UOM_REF_PATH)
df = df.merge(uom_ref_df, on="hs6", how="left")

df["uom_status"] = "OK"
df.loc[(df["has_quantity"]) & (df["quantity"] == 0), "uom_status"] = "Error"
df.loc[
    (df["expected_uom_code"].notna()) &
    (df["uom_code"] != df["expected_uom_code"]) &
    (df["uom_status"] == "OK"),
    "uom_status"
] = "Mismatch"

print("► Computing value outlier flags...")
bounds = compute_outlier_bounds(df)
df = df.merge(bounds, on="hs8", how="left")

df["is_outlier"] = False
mask_bounds = df["lower_bound_hs8"].notna() & df["upper_bound_hs8"].notna()
df.loc[mask_bounds, "is_outlier"] = (
    (df.loc[mask_bounds, "value"] < df.loc[mask_bounds, "lower_bound_hs8"]) |
    (df.loc[mask_bounds, "value"] > df.loc[mask_bounds, "upper_bound_hs8"])
)

print("► Final column ordering...")
for col in FINAL_COLS:
    if col not in df.columns:
        df[col] = np.nan

df_clean = df[FINAL_COLS].copy()

print("► Saving cleaned dataset...")
df_clean.to_csv(CLEAN_CSV, index=False)

print("► Creating PDF cleaning report...")
create_pdf_report(df_clean, AUDIT_PDF)

print("\n" + "=" * 80)
print("✅ DATA CLEANING COMPLETE")
print("=" * 80)
print(f"Clean dataset : {CLEAN_CSV}")
print(f"Cleaning PDF  : {AUDIT_PDF}")

display(df_clean.head())

02 DATA CLEANING
Clean CSV : /content/drive/MyDrive/canada-brazil-trade-report/data/clean/dataset_clean.csv
Audit PDF : /content/drive/MyDrive/canada-brazil-trade-report/reports/02_data_cleaning_report.pdf
► Combining import and export datasets...
► Standardizing column names and core fields...
► Creating date features...
► Standardizing numeric columns...
► Adding core flags...
► Joining HS mapping...
⚠️ Mapping file not found: /content/drive/MyDrive/canada-brazil-trade-report/data/hs_mapping.xlsx
► Auditing UOM...
⚠️ UOM reference file not found: /content/drive/MyDrive/canada-brazil-trade-report/data/t2026_uom.xlsx
► Computing value outlier flags...
► Final column ordering...
► Saving cleaned dataset...
► Creating PDF cleaning report...
✅ PDF saved: /content/drive/MyDrive/canada-brazil-trade-report/reports/02_data_cleaning_report.pdf

✅ DATA CLEANING COMPLETE
Clean dataset : /content/drive/MyDrive/canada-brazil-trade-report/data/clean/dataset_clean.csv
Cleaning PDF  : /content/drive/

,period_m,year,month,quarter,flow,province,country,description,hs2,hs4,hs6,hs8,hs10,section,section_name,chapter_name,value,quantity,uom_text,uom_code,uom_status,has_quantity,is_ch98,is_suppressed,is_outlier,median_value_hs8,lower_bound_hs8,upper_bound_hs8
0,2025-01,2025,1,1,Import,Newfoundland and Labrador,Brazil,"Plates, sheets and strip, of vulcanized cellul...",00,0000,000000,00000040,0000004008,NaN,NaN,NaN,173,10,Weight in kilograms,KGM,OK,True,False,False,False,"1,135.00",0.00,"24,358.00"
1,2024-02,2024,2,1,Import,Newfoundland and Labrador,Brazil,"Other AC motors, multi-phase, of an output exc...",00,0000,000000,00000085,0000008501,NaN,NaN,NaN,2558388,2,Number,NMB,OK,True,False,False,True,"1,864.50",0.00,"34,311.00"
2,2025-03,2025,3,1,Import,Newfoundland and Labrador,Brazil,"Parts, of hand tools, nes",00,0000,000000,00000084,0000008467,NaN,NaN,NaN,46,0,Blank,BLANK,OK,False,False,False,False,"3,527.00",0.00,"54,860.00"
3,2025-10,2025,10,4,Import,Newfoundland and Labrador,Brazil,"Screws, whether/not with their nuts or washers...",00,0000,000000,00000073,0000007318,NaN,NaN,NaN,8,0,Weight in kilograms,KGM,Error,True,False,False,False,731.00,0.00,"11,567.00"
4,2025-11,2025,11,4,Import,Newfoundland and Labrador,Brazil,"Other AC motors, single-phase, power >750 W bu...",00,0000,000000,00000085,0000008501,NaN,NaN,NaN,1209,1,Number,NMB,OK,True,False,False,False,"1,864.50",0.00,"34,311.00"
